# Day 17 — Production Testing, Validation & Quality Assurance

The platform predicts, retrieves, explains, and reports — Day 17 makes it **reliable**: a **91-test**
suite that validates every layer, split into a **fast lane (53)** that runs in CI on tiny fixtures and a
**heavy lane (38)** that needs the real data, models, and 407 MB index and runs locally.

**The disciplines (all shown below):**
- **Test properties, not constants** — the model *clears its baseline* and *reproduces*, SHAP is
  *additive*, the schema *holds* — never `f1 == 0.6664`, which a library patch would break.
- **Be honest about ground truth** — the model has real labels (legit metrics); retrieval has none, so
  proxy metrics, not a fabricated Precision@K.
- **Honest CI** — data/models/index are git-ignored, so CI runs only the fast lane; heavy tests run here.

*(This notebook runs the fast lane + data validation live, and displays the heavy lane's JSON reports —
generated by `pytest -m heavy`. It loads no models itself, so it's quick.)*

In [1]:
import os, sys, json, subprocess, time
os.environ["OMP_NUM_THREADS"] = "1"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
PY = sys.executable
print("repo:", os.getcwd())

repo: /Users/Yash/AI-Ventutre-Intelligence-Platform


## 1 — The suite at a glance: fast vs heavy

Markers, not directories, draw the line. `pytest --co` collects the whole suite; `-m "not heavy"` is the
CI lane, `-m heavy` is the local lane.

In [2]:
def _collect(marker=None):
    cmd = [PY, "-m", "pytest", "--co", "-q"] + (["-m", marker] if marker else [])
    out = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "OMP_NUM_THREADS": "1"})
    line = [l for l in out.stdout.splitlines() if "collected" in l or "selected" in l]
    return line[-1] if line else out.stdout.strip().splitlines()[-1]

print("ALL   :", _collect())
print("FAST  :", _collect("not heavy"))
print("HEAVY :", _collect("heavy"))

ALL   : 91 tests collected in 1.34s


FAST  : 53/91 tests collected (38 deselected) in 1.10s


HEAVY : 38/91 tests collected (53 deselected) in 1.04s


## 2 — The fast lane (what CI runs on every push)

Fixture-based, no real data/models — so it runs in seconds on a GitHub runner. This is the exact command
in `.github/workflows/ci.yml`.

In [3]:
t = time.perf_counter()
out = subprocess.run([PY, "-m", "pytest", "-m", "not heavy", "-q"],
                     capture_output=True, text=True, env={**os.environ, "OMP_NUM_THREADS": "1"})
print(out.stdout.strip().splitlines()[-1], f"  ({time.perf_counter()-t:.1f}s wall)")

53 passed, 38 deselected, 2 warnings in 4.28s   (5.3s wall)


## 3 — Data validation (live, on the REAL dataset)

The validation logic is fixture-tested in CI; here we run it on the real 331,464-row dataset + 865-wide
feature matrix. Note **severity**: critical failures invalidate the data; downstream-handled quality
issues are *warnings*.

In [4]:
import pandas as pd, numpy as np, scipy.sparse as sp
from tests.test_data_validation import build_data_validation_report

df = pd.read_csv("data/processed/kickstarter_processed.csv")
X = sp.load_npz("artifacts/fused_features/X_train_selected.npz")
y = np.load("artifacts/fused_features/y_train.npy")
rep = build_data_validation_report(df, X, y)

print(f"passed={rep['passed']}  rows={rep['n_rows']:,}  critical_failed={rep['n_critical_failed']}  warnings={rep['n_warnings']}")
for c in rep["checks"]:
    flag = "OK  " if c["status"] == "PASS" else f"{c['severity'].upper():8}"
    print(f"  [{flag}] {c['name']:<26} {c['detail']}")

passed=True  rows=331,464  critical_failed=0  warnings=1
  [OK  ] required_columns           missing=[]
  [OK  ] target_binary              target must be {0,1}
  [OK  ] target_no_missing          no NaN labels
  [OK  ] goal_positive              all goals > 0
  [WARNING ] name_clean_present         4 NaN cleaned titles (handled downstream by fillna(''))
  [OK  ] main_category_cardinality  15 categories
  [OK  ] no_duplicate_rows          0 dup rows
  [OK  ] feature_matrix_2d          shape=(265171, 865)
  [OK  ] feature_no_nan_inf         no NaN/inf in features
  [OK  ] feature_target_aligned     X rows 265171 vs y 265171
  [OK  ] y_binary                   labels {0,1}


## 4 — The heavy lane's validation reports

Generated by `pytest -m heavy` (model / retrieval / explainability / benchmark). We display the real
artifacts rather than re-running the minutes-long heavy suite here.

In [5]:
from pathlib import Path
def show(path, title):
    p = Path("artifacts/testing") / path
    print(f"### {title}")
    if not p.exists():
        print("  (run `pytest -m heavy` to generate)\n"); return None
    return json.loads(p.read_text())

# --- model validation ---
m = show("model_validation.json", "Model validation (calibrated LightGBM)")
if m:
    print(f"  passed={m['passed']}  roc_auc_gap={m['roc_auc_gap']} (no overfit)")
    for k, v in m["checks"].items(): print(f"    {k:<26} {v}")
    print(f"  test: PR-AUC {m['test_metrics']['pr_auc']}  ROC-AUC {m['test_metrics']['roc_auc']}  vs baseline {m['baseline_logreg']['pr_auc']}")
    print(f"  deferred: {m['deferred']}\n")

### Model validation (calibrated LightGBM)
  passed=True  roc_auc_gap=0.0404 (no overfit)
    pr_auc>=0.68               True
    roc_auc>=0.77              True
    brier<=0.20                True
    f1>=0.63                   True
    recall>=0.75               True
    beats_logreg_baseline      True
    no_overfit_gap<0.10        True
  test: PR-AUC 0.6883  ROC-AUC 0.7763  vs baseline 0.6264
  deferred: {'neural_network': 'no NN trained (torch is SBERT-only) — validation deferred'}



In [6]:
# --- retrieval validation (honest proxy metrics, NO Precision@K) ---
r = show("retrieval_validation.json", "Retrieval validation (proxy metrics)")
if r:
    print(f"  passed={r['passed']}  n_queries={r['n_queries']}")
    cc = r["category_consistency"]["k5"]; kn = r["knn_outcome"]["k5"]
    print(f"    category lift@5 : {cc['lift']}x  (consistency {cc['consistency']:.0%} vs base {cc['base_rate']:.0%})")
    print(f"    kNN ROC-AUC@5   : {kn['roc_auc']}  (beats majority by {kn['lift_over_majority']})")
    print(f"    MRR@5           : {r['mrr_category']['k5']}")
    print(f"  {r['honesty_note']}\n")

# --- explainability validation (SHAP additivity) ---
e = show("explainability_validation.json", "Explainability validation (SHAP additivity)")
if e:
    print(f"  passed={e['passed']}  additivity_error={e['additivity_error']:.2e}  (base + Σφ ≈ model margin)")
    print(f"  {e['honesty_note']}\n")

# --- performance benchmark ---
b = show("benchmark.json", "Performance benchmark")
if b:
    print(f"  memory RSS: {b['memory_rss_mb']} MB   CPU: {b['cpu_percent']}%   ({b['n_cpus']} cores)")
    for k, v in b["latency_ms"].items(): print(f"    {k:<16} p50={v['p50']:>8.1f} ms   p95={v['p95']:>8.1f} ms")

### Retrieval validation (proxy metrics)
  passed=True  n_queries=300
    category lift@5 : 7.13x  (consistency 77% vs base 11%)
    kNN ROC-AUC@5   : 0.6849  (beats majority by 0.05)
    MRR@5           : 0.8541
  proxy metrics vs real labels — no Precision@K (no similarity ground truth)

### Explainability validation (SHAP additivity)
  passed=True  additivity_error=3.12e-08  (base + Σφ ≈ model margin)
  SHAP explains the uncalibrated log-odds margin; calibration is a monotonic squash

### Performance benchmark
  memory RSS: 1092.4 MB   CPU: 92.5%   (11 cores)
    predict          p50=    11.4 ms   p95=    13.0 ms
    explain          p50=    32.2 ms   p95=    32.6 ms
    retrieve         p50=    15.0 ms   p95=    15.5 ms
    report_founder   p50=  2149.9 ms   p95=  2220.2 ms
    analyze          p50=  3397.7 ms   p95=  5766.7 ms


## Summary — the pattern, and the honesty

**Layer-by-layer validation of behaviour.** 91 tests: a fast fixture lane in CI, a heavy artifact lane
locally. Every test asserts a **property or threshold** — clears the baseline, reproduces exactly, SHAP
additive to 3e-8, schema valid, deferred-honesty fields survive end to end — never a frozen decimal. And
the discipline is honesty about ground truth (proxy metrics where no labels exist) and about
infrastructure (CI runs what it can; coverage reported, not gated).

> **Interview soundbite.** *"Day 17 makes the platform reliable with 91 tests split by marker: 53 fast
> fixture tests in CI, 38 heavy artifact tests locally, because the data and a 407 MB index are
> git-ignored. Every test asserts a property, not a constant — the model clears its baseline and
> reproduces, SHAP is additive to 3e-8, the API rejects fabricated fields with a 422, and the
> deferred-honesty fields survive end to end. Validating the real dataset even caught 4 null cleaned
> titles, which I classified as a downstream-handled warning rather than failing a working dataset. The
> throughline is honesty: proxy metrics where there's no ground truth, and coverage reported without
> gating on a number the fast lane can't back."*